# google_genai smoke test

Smoke test for `google-genai` (the Google Gen AI SDK). Import name is
`google.genai` (NOT `google_genai`). All examples run offline: client
construction with a fake API key, typed request models, validation, and the
error hierarchy — no generate call is made.

In [ ]:
import pkgutil, importlib, importlib.util
import google.genai as _pkg_under_test  # import name for the google-genai dist

# The wheel ships its own test suite (google.genai.tests.*), which requires replay
# fixtures + env vars, plus a stray _test_api_client helper — both are excluded:
# they are not part of the importable API surface.
EXCLUDED = (f"{_pkg_under_test.__name__}.tests", f"{_pkg_under_test.__name__}._test_api_client")
# The local tokenizer modules need the optional `local-tokenizer` extra (sentencepiece); skip them without it.
if importlib.util.find_spec("sentencepiece") is None:
    EXCLUDED += (f"{_pkg_under_test.__name__}.local_tokenizer", f"{_pkg_under_test.__name__}._local_tokenizer_loader")

failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    if module_info.name.startswith(EXCLUDED):
        continue
    try:
        importlib.import_module(module_info.name)
    except Exception as e:
        failed.append((module_info.name, type(e).__name__, str(e)))

if failed:
    for name, etype, msg in failed:
        print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all non-test submodules of {_pkg_under_test.__name__} imported cleanly")

In [ ]:
# Example 1: Client construction with an explicit API key (no network until a call).
from google import genai

client = genai.Client(api_key="fake-api-key-for-offline-smoke-test")
print(f"client: {type(client).__name__}")
surfaces = ["models", "chats", "files", "caches", "batches", "tunings"]
for name in surfaces:
    print(f"  client.{name}: {type(getattr(client, name)).__name__}")
missing = [n for n in surfaces if not hasattr(client, n)]
assert not missing, f"missing client surfaces: {missing}"
assert not client.vertexai, "api-key client must target the Gemini API, not Vertex"

In [ ]:
# Example 2: Typed request models — Part / Content construction.
from google.genai import types

part = types.Part.from_text(text="Hello from the smoke test")
content = types.Content(role="user", parts=[part])

print(f"part.text: {part.text}")
print(f"content.role: {content.role}")
print(f"parts: {len(content.parts)}")
assert part.text == "Hello from the smoke test"
assert content.role == "user"
assert content.parts[0] is part

In [ ]:
# Example 3: GenerateContentConfig — pydantic validation accepts good input,
# rejects bad input.
import pydantic

from google.genai import types

cfg = types.GenerateContentConfig(temperature=0.5, top_p=0.9, max_output_tokens=64)
print(f"temperature={cfg.temperature} top_p={cfg.top_p} max_output_tokens={cfg.max_output_tokens}")
assert cfg.temperature == 0.5 and cfg.max_output_tokens == 64

try:
    types.GenerateContentConfig(temperature="definitely-not-a-number")
    raise AssertionError("expected pydantic.ValidationError")
except pydantic.ValidationError as e:
    print(f"ValidationError raised as expected ({len(e.errors())} error)")

In [ ]:
# Example 4: JSON round trip — model_dump_json / model_validate_json.
from google.genai import types

content = types.Content(role="model", parts=[types.Part.from_text(text="pong")])
payload = content.model_dump_json(exclude_none=True)
print(f"payload: {payload}")

restored = types.Content.model_validate_json(payload)
print(f"restored role: {restored.role}, text: {restored.parts[0].text}")
assert restored == content
assert restored.parts[0].text == "pong"

In [ ]:
# Example 5: Error hierarchy — the APIError family used by all API surfaces.
from google.genai import errors

print(f"APIError: {errors.APIError}")
print(f"ClientError mro: {[b.__name__ for b in errors.ClientError.__mro__[:3]]}")
assert issubclass(errors.ClientError, errors.APIError)
assert issubclass(errors.ServerError, errors.APIError)

err = errors.APIError(404, {"error": {"message": "not found", "status": "NOT_FOUND"}})
print(f"constructed: code={err.code} message={err.message!r}")
assert err.code == 404